In [ ]:
import sys
import sys
sys.path.append("..")
from topofinger import config
from topofinger.hcp_IO import load_scaffold_cortical, load_fc_cortical, load_scaffold_schaefer300, load_scaffold, save_distances_subjects, save_correlations_subjects, save_correlations_subjects_fast
from topofinger.utils import apply_threshold_to_nested_dict, threshold_matrix
from topofinger.fingerprinting import compute_group_consistency, compute_differential_power_fast as compute_differential_power
from topofinger.utils_subnets import compute_subnet_density, compute_subnet_mean
from topofinger.viz import plot_connectivity_matrix_functional_subnetworks

import pickle as pk
from tqdm.notebook import tqdm
import numpy as np

import matplotlib.pyplot as plt
from pingouin import compute_effsize
from scipy.stats import pearsonr, zscore
from sklearn.preprocessing import LabelEncoder

from collections import Counter



%load_ext autoreload
%autoreload 2

## Load the data

In [ ]:
data_dir = config.HCP_1200 + '/NoGSR_LowPassOnly/'
scaffold_dir = 'scaffolds/whole/'
fc_dir = 'fc/whole/'

#scaffolds
sc = load_scaffold_cortical(data_dir, config.N_CORTICAL)  # 300 cortical regions

#fc
fc = load_fc_cortical(data_dir, config.N_CORTICAL)
fc_th = apply_threshold_to_nested_dict(fc,1)

#open list of regions from txt
subnet_list = []
with open(config.HCP_100 + '/schaefer300/Schaefer2018_300Parcels_7Networks_order.lut','r') as f:
    for line in f:
        elements = line.strip().split(' ')
        subnet_list.append(elements[4].split('_')[2])


#encode subnet names as numbers
label_encoder = LabelEncoder()
subnet_num_list = label_encoder.fit_transform(subnet_list)

In [ ]:
#binarize the lists of scaffolds and thresholded fc

s0 = np.array([(sub[0]>0).astype(int) for sub in sc.values()])
s1 = np.array([(sub[1]>0).astype(int) for sub in sc.values()])

fc0 = np.array([(sub[0]>0).astype(int) for sub in fc_th.values()])
fc1 = np.array([(sub[1]>0).astype(int) for sub in fc_th.values()])

In [ ]:
#sum the binarized connectivity matrix, the results gives each edge a weight
# that correspond to the % of individuals that have it

s0_all = np.sum(s0,axis=0)
s1_all = np.sum(s1,axis=0)

fc0_all = np.sum(fc0,axis=0)
fc1_all = np.sum(fc1,axis=0)

In [ ]:
#open list of regions from txt
subnet_list = []
with open(config.HCP_100 + '/schaefer300/Schaefer2018_300Parcels_7Networks_order.lut','r') as f:
    for line in f:
        elements = line.strip().split(' ')
        subnet_list.append(elements[4].split('_')[2])

#encode subnet names as numbers
label_encoder = LabelEncoder()
subnet_num_list = label_encoder.fit_transform(subnet_list)


subnet_dict = dict(zip(label_encoder.classes_, range(0,subnet_num_list.max()+1)))


subnet_order = np.argsort(subnet_num_list, kind='stable')

counts = Counter(subnet_num_list)
ordered_counts = dict(sorted(counts.items()))
num_roi_per_subnet = [counts[i] for i in ordered_counts.keys()]
subnet_ranges = []
start_idx = 0
for lenght in num_roi_per_subnet:
    subnet_ranges.append((start_idx, start_idx+lenght))
    start_idx += lenght

from matplotlib.colors import ListedColormap

# ColorBrewer Set2, first 7 classes (same colors as palettable colorbrewer Set2_7)
colors = [ListedColormap(plt.get_cmap("Set2").colors[:7])(i) for i in range(0,7)]

In [ ]:
s0_all[s0_all!=0] = 1
s1_all[s1_all!=0] = 1
fc0_all[fc0_all!=0] = 1
fc1_all[fc1_all!=0] = 1


plot_connectivity_matrix_functional_subnetworks(s0_all, subnet_dict.keys(), subnet_order, colors, subnet_ranges)
plot_connectivity_matrix_functional_subnetworks(s1_all, subnet_dict.keys(), subnet_order, colors, subnet_ranges)

plot_connectivity_matrix_functional_subnetworks(fc0_all, subnet_dict.keys(), subnet_order, colors, subnet_ranges)
plot_connectivity_matrix_functional_subnetworks(fc1_all, subnet_dict.keys(), subnet_order, colors, subnet_ranges)


In [ ]:
s0 = np.array([sub[0] for sub in sc.values()])
s1 = np.array([sub[1] for sub in sc.values()])

scaffolds_all = np.sum(np.concatenate((s0, s1), axis=0), axis=0)
#scaffolds_all = (scaffolds_all > 100).astype(int)

fc0 = np.array([sub[0] for sub in fc_th.values()])
fc1 = np.array([sub[1] for sub in fc_th.values()])

fc_all = np.sum(np.concatenate((fc0, fc1), axis=0), axis=0)
#fc_all = (fc_all > 50).astype(int)

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(scaffolds_all, cmap='viridis')
axs[0].set_title('Scaffolds All')
axs[1].imshow(fc_all, cmap='viridis')
axs[1].set_title('FC All')
plt.show()

In [ ]:
#open list of regions from txt
subnet_list = []
with open(config.HCP_100 + '/schaefer300/Schaefer2018_300Parcels_7Networks_order.lut','r') as f:
    for line in f:
        elements = line.strip().split(' ')
        subnet_list.append(elements[4].split('_')[2])



#encode subnet names as numbers
label_encoder = LabelEncoder()
subnet_num_list = label_encoder.fit_transform(subnet_list)


subnet_dict = dict(zip(label_encoder.classes_, range(0,subnet_num_list.max()+1)))


subnet_order = np.argsort(subnet_num_list, kind='stable')

counts = Counter(subnet_num_list)
ordered_counts = dict(sorted(counts.items()))
num_roi_per_subnet = [counts[i] for i in ordered_counts.keys()]
subnet_ranges = []
start_idx = 0
for lenght in num_roi_per_subnet:
    subnet_ranges.append((start_idx, start_idx+lenght))
    start_idx += lenght

from matplotlib.colors import ListedColormap

# ColorBrewer Set2, first 7 classes (same colors as palettable colorbrewer Set2_7)
colors = [ListedColormap(plt.get_cmap("Set2").colors[:7])(i) for i in range(0,7)]

In [ ]:
plot_connectivity_matrix_functional_subnetworks(scaffolds_all, subnet_dict.keys(), subnet_order, colors, subnet_ranges)
plot_connectivity_matrix_functional_subnetworks(fc_all, subnet_dict.keys(), subnet_order, colors, subnet_ranges)

## Group consistency

In [ ]:
group_consistency_fc = compute_group_consistency(fc_th, z=True)
group_consistency_sc = compute_group_consistency(sc, z=True)


fig, axs = plt.subplots(1, 2, figsize=(10, 5))

# Plot group consistency of FC
axs[0].imshow(threshold_matrix(group_consistency_fc, 0.5))
axs[0].set_title('Group Consistency FC')

# Plot group consistency of SC
axs[1].imshow(threshold_matrix(group_consistency_sc, 0.5))
axs[1].set_title('Group Consistency Scaffolds')

plt.show()

In [ ]:
group_consistency_fc.shape

In [ ]:
group_consistency_fc_density = compute_subnet_density(group_consistency_fc, subnet_list)
group_consistency_sc_density = compute_subnet_density(group_consistency_sc, subnet_list)

fig, axs = plt.subplots(1, 2, figsize=(11, 5))

# Plot group consistency of FC
axs[0].imshow(group_consistency_fc_density)
axs[0].set_title('Group Consistency FC, subnet densities')

# Plot group consistency of SC
axs[1].imshow(group_consistency_sc_density)
axs[1].set_title('Group Consistency Scaffolds, subnet densities')

axs[0].set_xticks(range(len(label_encoder.classes_)), label_encoder.classes_, rotation=45)
axs[0].set_yticks(range(len(label_encoder.classes_)), label_encoder.classes_, rotation=45)
axs[1].set_xticks(range(len(label_encoder.classes_)), label_encoder.classes_, rotation=45)
axs[1].set_yticks(range(len(label_encoder.classes_)), label_encoder.classes_, rotation=45)


plt.show()

For the group consistency the z score doesn't change results
If we compute the mean value instead of the density in each area we see a more spread distribution of scaffolds links across regions

## Differential power

In [ ]:
differential_power_fc = compute_differential_power(fc_th, z=True)
differential_power_sc = compute_differential_power(sc, z=True)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5))

# Plot group consistency of FC
axs[0].imshow(threshold_matrix(differential_power_fc, 0.5))
axs[0].set_title('Differential power FC')

# Plot group consistency of SC
axs[1].imshow(threshold_matrix(differential_power_sc, 0.5))
axs[1].set_title('Differential power Scaffolds')

plt.show()

In [ ]:
differential_power_fc_density = compute_subnet_density(differential_power_fc, subnet_list)
differential_power_sc_density = compute_subnet_density(differential_power_sc, subnet_list)

fig, axs = plt.subplots(1, 2, figsize=(11, 5))

# Plot group consistency of FC
axs[0].imshow(differential_power_fc_density)
axs[0].set_title('Differential power FC, subnet densities')

# Plot group consistency of SC
axs[1].imshow(differential_power_sc_density)
axs[1].set_title('Differential power Scaffolds, subnet densities')

axs[0].set_xticks(range(len(label_encoder.classes_)), label_encoder.classes_, rotation=45)
axs[0].set_yticks(range(len(label_encoder.classes_)), label_encoder.classes_, rotation=45)
axs[1].set_xticks(range(len(label_encoder.classes_)), label_encoder.classes_, rotation=45)
axs[1].set_yticks(range(len(label_encoder.classes_)), label_encoder.classes_, rotation=45)


plt.show()

In [ ]:
edge_importance_all = {
    "differential_power_fc": differential_power_fc,
    "differential_power_sc": differential_power_sc,
    "group_consistency_fc": group_consistency_fc,
    "group_consistency_sc": group_consistency_sc
}

with open(config.RESULTS + '/edge_importance_all.pck', 'wb') as f:
    pk.dump(edge_importance_all, f)

In [ ]:
edge_importance = {
    "differential_power_fc_density": differential_power_fc_density,
    "differential_power_sc_density": differential_power_sc_density,
    "group_consistency_fc_density": group_consistency_fc_density,
    "group_consistency_sc_density": group_consistency_sc_density
}

with open(config.RESULTS + '/edge_importance.pck', 'wb') as f:
    pk.dump(edge_importance, f)

In [ ]:
differential_power_fc_mean = compute_subnet_mean(differential_power_fc, subnet_list)
differential_power_sc_mean = compute_subnet_mean(differential_power_sc, subnet_list)

fig, axs = plt.subplots(1, 2, figsize=(11, 5))

# Plot group consistency of FC
axs[0].imshow(differential_power_fc_mean)
axs[0].set_title('Differential power FC, subnet means')

# Plot group consistency of SC
axs[1].imshow(differential_power_sc_mean)
axs[1].set_title('Differential power Scaffolds, subnet means')

axs[0].set_xticks(range(len(label_encoder.classes_)), label_encoder.classes_, rotation=45)
axs[0].set_yticks(range(len(label_encoder.classes_)), label_encoder.classes_, rotation=45)
axs[1].set_xticks(range(len(label_encoder.classes_)), label_encoder.classes_, rotation=45)
axs[1].set_yticks(range(len(label_encoder.classes_)), label_encoder.classes_, rotation=45)

plt.show()

## Behavioral scores

In [ ]:
file_path = config.BEHAV_CSV
subjects = fc[300].keys()
# Load the CSV file into a DataFrame
df = pd.read_csv(file_path)
df['Subject'] = df['Subject'].astype(str)

#filter subjects
filtered_df = df[df['Subject'].isin(subjects)]
filtered_df.set_index('Subject', inplace=True)

#filter only numerical values
filtered_df = filtered_df.select_dtypes(include='number')

#order df according to subject list
new_idx = list(fc[300].keys())
filtered_df = filtered_df.reindex(new_idx)

#remove columns with nans
columns_with_nans = filtered_df.columns[filtered_df.isna().any()]
filtered_df = filtered_df.drop(columns=columns_with_nans)



filtered_df

In [ ]:
mapping_behavioural_scores={'Episodic memory' : ['PicSeq_Unadj'],
 'Executive function' : ['Flanker_Unadj','CardSort_Unadj'],
 'Fluid Intelligence':['PMAT24_A_CR','PMAT24_A_SI','PMAT24_A_RTCR'],
 'Language': ['ReadEng_Unadj','PicVocab_Unadj'],
 'Processing Speed' : ['ProcSpeed_Unadj'],
 'Self-regulation/Inpulsivity': ['DDisc_AUC_200','DDisc_AUC_40K'],
 'Spatial Orientation': ['VSPLOT_TC','VSPLOT_CRTE','VSPLOT_OFF'],
 'Sustained visual attention': ['SCPT_TP','SCPT_TN','SCPT_FP','SCPT_FN','SCPT_TPRT','SCPT_SEN','SCPT_SPEC','SCPT_LRNR'], ## This is slightly different
 'Verbal episodic memory': ['IWRD_TOT','IWRD_RTC'],
 'Working Memory': ['ListSort_Unadj']}

In [ ]:
pca = PCA(n_components=1)

beh_scores = pd.DataFrame()

beh_scores.index = filtered_df.index

for key in mapping_behavioural_scores.keys():
    X = filtered_df[mapping_behavioural_scores[key]]
    X = pca.fit_transform(X)
    
    beh_scores[key] = list(X.flatten())
    
beh_scores